# 82 — Analyze 20-action fork acquisition

This notebook supports partial inspection after workers 0/1 and full analysis after all four. It also compares roots retained exactly from the 10-action manifest.


In [ ]:
EXTRAS = 'analysis'
SETUP_ENV = True
import urllib.request
exec(urllib.request.urlopen(
    'https://raw.githubusercontent.com/ArjunS07/cs159-sp26/main/'
    'pnp-vla/scripts/colab_bootstrap.py').read().decode())


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from pnp.qplanning_fork_pilot import load_fork_pilot_results
from pnp.qplanning_fork20_pilot import load_fork20_pilot_results

trees20, summary20 = load_fork20_pilot_results()
trees10, summary10 = load_fork_pilot_results()
display(summary20)
completion = (trees20.groupby('strategy', observed=True)
              .agg(expected=('candidate_group_id', 'size'),
                   complete=('complete', 'sum'),
                   branches=('branches', 'sum')).reset_index())
display(completion)


In [ ]:
order = ['random', 'u20', 'failure']
plot = summary20.set_index('strategy').loc[order]
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].bar(order, plot.mixed_outcome_trees_pct)
axes[0].set(title='20-action intervention: mixed trees', ylabel='Mixed-outcome trees (%)')
axes[1].bar(order, plot.oracle_gain_over_stock_pp)
axes[1].axhline(0, color='black', linewidth=1)
axes[1].set(title='20-action intervention: oracle gain', ylabel='Any-success minus stock (pp)')
for axis in axes: axis.grid(axis='y', alpha=.25)
fig.tight_layout(); plt.show()


In [ ]:
keys = ['strategy', 'source_rollout_id', 'chunk_idx']
left = trees10[trees10.complete].copy()
right = trees20[(trees20.complete) & (trees20.paired_with_10action_tree)].copy()
paired = left.merge(right, on=keys, suffixes=('_10', '_20'), validate='one_to_one')
paired_summary = (paired.groupby('strategy', observed=True)
    .agg(paired_trees=('candidate_group_id_20', 'size'),
         mixed_10action_pct=('mixed_outcomes_10', lambda x: 100*x.mean()),
         mixed_20action_pct=('mixed_outcomes_20', lambda x: 100*x.mean()),
         oracle_10action_pp=('any_success_10', lambda x: 100*x.mean()),
         oracle_20action_pp=('any_success_20', lambda x: 100*x.mean()))
    .reset_index())
paired_summary['oracle_10action_pp'] -= (paired.groupby('strategy').stock_success_10.mean().to_numpy()*100)
paired_summary['oracle_20action_pp'] -= (paired.groupby('strategy').stock_success_20.mean().to_numpy()*100)
display(paired_summary)
print({'paired_complete_trees': len(paired),
       'interpretation': 'Use this matched table for the clean 10-vs-20 horizon comparison.'})


Do not interpret tiny partial differences as final effects. The main signal is whether 20 actions materially increase the matched mixed-tree rate and available oracle gain.
